In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

print("Notebook 05 started")
print("Output directory:", OUTPUT_DIR)

Notebook 05 started
Output directory: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs


In [8]:
model_comparison = pd.read_csv(
    OUTPUT_DIR / "model_comparison.csv"
)

risk_summary = pd.read_csv(
    OUTPUT_DIR / "risk_tier_validation.csv"
)

evaluation = pd.read_csv(
    OUTPUT_DIR / "evaluation_predictions.csv"
)

model_features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

print("Model comparison:", model_comparison.shape)
print("Risk summary:", risk_summary.shape)
print("Evaluation:", evaluation.shape)
print("Features:", model_features.shape)

Model comparison: (3, 7)
Risk summary: (4, 3)
Evaluation: (1384, 7)
Features: (5516, 27)


In [9]:
model_comparison = model_comparison.sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

selected_model_name = model_comparison.loc[
    0,
    "model"
]

print("Selected model:", selected_model_name)

display(model_comparison)

Selected model: Logistic Regression


,model,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.702674,0.671655,0.648126,0.615764,0.649351,0.632111
1,Extra Trees,0.697502,0.653645,0.644498,0.612346,0.644156,0.627848
2,Random Forest,0.691603,0.646889,0.634825,0.605598,0.618182,0.611825


In [10]:
exclude_columns = [
    "episode_id",
    "teleconsult_id",
    "patient_id",
    "consult_date",
    "cohort",
    "lost_to_followup_label"
]

feature_columns = [
    c for c in model_features.columns
    if c not in exclude_columns
]

X = model_features[feature_columns].copy()

print("Explainable features:", len(feature_columns))
display(X.head())

Explainable features: 21


,age,distance_km,duration_min,review_due_days,medicine_required,test_required,review_required,care_requirements_count,complex_care_episode,previous_teleconsultations,...,days_since_first_recorded_consult,consultation_frequency,previous_care_requirements,poor_connectivity,long_distance,low_digital_access,elderly_living_alone,consult_month,consult_dayofweek,is_weekend
0,47,29.7,15,21.0,1,1,0,2,1,1,...,30,0.033333,0.0,1,1,1,0,5,1,0
1,45,33.3,18,21.0,0,0,1,1,0,2,...,87,0.022989,0.0,0,1,0,0,7,2,0
2,45,27.6,20,7.0,0,0,1,1,0,3,...,107,0.028037,1.0,0,1,0,0,8,1,0
3,46,25.7,16,30.0,1,1,1,3,1,0,...,0,0.000000,0.0,0,1,0,1,6,1,0
4,18,17.2,19,14.0,1,1,1,3,1,0,...,0,0.000000,0.0,0,1,0,0,4,4,0


In [11]:
# CELL 5 — Model-based feature importance

from sklearn.impute import SimpleImputer

X_clean = X.copy()

imputer = SimpleImputer(strategy="median")

X_clean_array = imputer.fit_transform(X_clean)

X_clean = pd.DataFrame(
    X_clean_array,
    columns=feature_columns,
    index=X.index
)

importance_table = None

if selected_model_name in [
    "Random Forest",
    "Extra Trees"
]:
    from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

    development_mask = (
        model_features["cohort"] == "DEVELOPMENT"
    )

    X_dev = X_clean.loc[development_mask]
    y_dev = (
        model_features.loc[
            development_mask,
            "lost_to_followup_label"
        ]
        .astype(int)
    )

    if selected_model_name == "Random Forest":
        importance_model = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    else:
        importance_model = ExtraTreesClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

    importance_model.fit(
        X_dev,
        y_dev
    )

    importance_table = pd.DataFrame({
        "feature": feature_columns,
        "importance": importance_model.feature_importances_
    }).sort_values(
        "importance",
        ascending=False
    )

else:
    print(
        "Selected model is Logistic Regression. "
        "Coefficient-based importance will be created next."
    )

display(
    importance_table.head(15)
    if importance_table is not None
    else pd.DataFrame()
)

Selected model is Logistic Regression. Coefficient-based importance will be created next.


""


In [12]:
# CELL 6 — Logistic Regression importance if applicable

if selected_model_name == "Logistic Regression":

    from sklearn.linear_model import LogisticRegression

    development_mask = (
        model_features["cohort"] == "DEVELOPMENT"
    )

    X_dev = X_clean.loc[development_mask]

    y_dev = (
        model_features.loc[
            development_mask,
            "lost_to_followup_label"
        ]
        .astype(int)
    )

    lr = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    lr.fit(
        X_dev,
        y_dev
    )

    importance_table = pd.DataFrame({
        "feature": feature_columns,
        "importance": np.abs(
            lr.coef_[0]
        )
    }).sort_values(
        "importance",
        ascending=False
    )

display(
    importance_table.head(15)
)

,feature,importance
7,care_requirements_count,0.773823
4,medicine_required,0.583481
12,consultation_frequency,0.208042
10,has_previous_teleconsult,0.182126
6,review_required,0.181973
17,elderly_living_alone,0.181085
16,low_digital_access,0.160440
8,complex_care_episode,0.128543
15,long_distance,0.070255
9,previous_teleconsultations,0.057026


In [13]:
# CELL 7 — SHAP analysis

try:
    import shap

    shap_available = True
    print("SHAP is available.")

except ImportError:
    shap_available = False
    print(
        "SHAP is not installed. "
        "Using model-based feature importance."
    )

c:\Users\RAAJARAPU SUSWIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP is available.


In [14]:
# CELL 8 — Install SHAP only if required


%pip install shap
    

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# CELL 9 — Create Final Explainability Table

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

# Load model features
model_features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

# Load model comparison
model_comparison = pd.read_csv(
    OUTPUT_DIR / "model_comparison.csv"
)

# Remove non-model columns
exclude_cols = [
    "episode_id",
    "patient_id",
    "consult_date",
    "cohort",
    "lost_to_followup_label"
]

feature_columns = [
    col for col in model_features.columns
    if col not in exclude_cols
]

# Development data only
development = model_features[
    model_features["cohort"] == "DEVELOPMENT"
].copy()

X = development[feature_columns]
y = development["lost_to_followup_label"]

# Fill missing numeric values
X = X.copy()

for col in X.columns:
    if X[col].dtype != "object":
        X[col] = X[col].fillna(X[col].median())

# Convert categorical columns if any exist
X = pd.get_dummies(
    X,
    drop_first=True
)

# Train the explainability model
from sklearn.ensemble import ExtraTreesClassifier

explain_model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

explain_model.fit(X, y)

# Create importance table
importance_table = pd.DataFrame({
    "feature": X.columns,
    "importance": explain_model.feature_importances_
})

importance_table = importance_table.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_table["importance_percent"] = (
    importance_table["importance"]
    / importance_table["importance"].sum()
    * 100
).round(2)

display(
    importance_table.head(15)
)

# Save output
importance_table.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "feature_importance.csv"
)

,feature,importance,importance_percent
0,care_requirements_count,0.124267,12.43
1,review_required,0.097452,9.75
2,medicine_required,0.086286,8.63
3,distance_km,0.081131,8.11
4,complex_care_episode,0.070544,7.05
5,long_distance,0.053395,5.34
6,previous_care_requirements,0.051020,5.10
7,duration_min,0.045985,4.60
8,review_due_days,0.044652,4.47
9,low_digital_access,0.042054,4.21


Saved: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs\feature_importance.csv


In [18]:
 import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

model_features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

exclude_cols = [
    "episode_id",
    "patient_id",
    "consult_date",
    "cohort",
    "lost_to_followup_label"
]

feature_columns = [
    col for col in model_features.columns
    if col not in exclude_cols
]

development = model_features[
    model_features["cohort"] == "DEVELOPMENT"
].copy()

X = development[feature_columns]
y = development["lost_to_followup_label"]

X = X.copy()

for col in X.columns:
    if X[col].dtype != "object":
        X[col] = X[col].fillna(X[col].median())

X = pd.get_dummies(
    X,
    drop_first=True
)

from sklearn.ensemble import ExtraTreesClassifier

explain_model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

explain_model.fit(X, y)

importance_table = pd.DataFrame({
    "feature": X.columns,
    "importance": explain_model.feature_importances_
})

importance_table = importance_table.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_table["importance_percent"] = (
    importance_table["importance"]
    / importance_table["importance"].sum()
    * 100
).round(2)

display(importance_table.head(15))

importance_table.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False
)

print("Saved feature_importance.csv")

,feature,importance,importance_percent
0,care_requirements_count,0.124267,12.43
1,review_required,0.097452,9.75
2,medicine_required,0.086286,8.63
3,distance_km,0.081131,8.11
4,complex_care_episode,0.070544,7.05
5,long_distance,0.053395,5.34
6,previous_care_requirements,0.051020,5.10
7,duration_min,0.045985,4.60
8,review_due_days,0.044652,4.47
9,low_digital_access,0.042054,4.21


Saved feature_importance.csv


In [19]:
# CELL 10 — Build Evaluation Intelligence

import pandas as pd
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

# Load required files
evaluation_predictions = pd.read_csv(
    OUTPUT_DIR / "evaluation_predictions.csv"
)

model_features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

print("Evaluation predictions:", evaluation_predictions.shape)
print("Model features:", model_features.shape)

display(evaluation_predictions.head())

Evaluation predictions: (1384, 7)
Model features: (5516, 27)


,episode_id,teleconsult_id,patient_id,consult_date,risk_probability,predicted_lost_to_followup,priority_tier
0,E0000002,TC0000002,P000004,2026-07-22,0.401178,0,MEDIUM
1,E0000003,TC0000003,P000004,2026-08-11,0.357278,0,MEDIUM
2,E0000009,TC0000009,P000008,2026-07-19,0.563782,1,MEDIUM
3,E0000014,TC0000014,P000010,2026-07-12,0.716091,1,HIGH
4,E0000018,TC0000018,P000014,2026-07-01,0.687923,1,HIGH


In [20]:
# CELL 10 — Build Evaluation Intelligence

import pandas as pd
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs"
)

# Load required files
evaluation_predictions = pd.read_csv(
    OUTPUT_DIR / "evaluation_predictions.csv"
)

model_features = pd.read_csv(
    OUTPUT_DIR / "model_features.csv"
)

print("Evaluation predictions:", evaluation_predictions.shape)
print("Model features:", model_features.shape)

display(evaluation_predictions.head())

Evaluation predictions: (1384, 7)
Model features: (5516, 27)


,episode_id,teleconsult_id,patient_id,consult_date,risk_probability,predicted_lost_to_followup,priority_tier
0,E0000002,TC0000002,P000004,2026-07-22,0.401178,0,MEDIUM
1,E0000003,TC0000003,P000004,2026-08-11,0.357278,0,MEDIUM
2,E0000009,TC0000009,P000008,2026-07-19,0.563782,1,MEDIUM
3,E0000014,TC0000014,P000010,2026-07-12,0.716091,1,HIGH
4,E0000018,TC0000018,P000014,2026-07-01,0.687923,1,HIGH


In [21]:
# CELL 11 — Add Episode Context

context_columns = [
    "episode_id",
    "patient_id",
    "consult_date",
    "age",
    "distance_km",
    "duration_min",
    "review_due_days",
    "medicine_required",
    "test_required",
    "review_required",
    "care_requirements_count",
    "complex_care_episode",
    "previous_teleconsultations",
    "has_previous_teleconsult",
    "previous_care_requirements",
    "poor_connectivity",
    "long_distance",
    "low_digital_access",
    "elderly_living_alone"
]

available_context = [
    col for col in context_columns
    if col in model_features.columns
]

evaluation_intelligence = evaluation_predictions.merge(
    model_features[available_context],
    on=["episode_id", "patient_id"],
    how="left"
)

print(
    "Evaluation intelligence shape:",
    evaluation_intelligence.shape
)

display(evaluation_intelligence.head())

Evaluation intelligence shape: (1384, 24)


,episode_id,teleconsult_id,patient_id,consult_date_x,risk_probability,predicted_lost_to_followup,priority_tier,consult_date_y,age,distance_km,...,review_required,care_requirements_count,complex_care_episode,previous_teleconsultations,has_previous_teleconsult,previous_care_requirements,poor_connectivity,long_distance,low_digital_access,elderly_living_alone
0,E0000002,TC0000002,P000004,2026-07-22,0.401178,0,MEDIUM,2026-07-22,45,33.3,...,1,1,0,2,1,0.0,0,1,0,0
1,E0000003,TC0000003,P000004,2026-08-11,0.357278,0,MEDIUM,2026-08-11,45,27.6,...,1,1,0,3,1,1.0,0,1,0,0
2,E0000009,TC0000009,P000008,2026-07-19,0.563782,1,MEDIUM,2026-07-19,66,20.3,...,1,2,1,2,1,2.0,0,1,0,0
3,E0000014,TC0000014,P000010,2026-07-12,0.716091,1,HIGH,2026-07-12,61,36.3,...,1,2,1,2,1,4.0,0,1,0,0
4,E0000018,TC0000018,P000014,2026-07-01,0.687923,1,HIGH,2026-07-01,36,11.9,...,1,3,1,0,0,0.0,1,1,1,0


In [22]:
# CELL 12 — Risk Tier

def assign_risk_tier(probability):

    if probability <= 0.30:
        return "LOW"

    elif probability <= 0.60:
        return "MEDIUM"

    elif probability <= 0.80:
        return "HIGH"

    else:
        return "VERY HIGH"


# Detect probability column
probability_candidates = [
    "risk_probability",
    "predicted_probability",
    "prediction_probability",
    "probability"
]

probability_column = None

for col in probability_candidates:
    if col in evaluation_intelligence.columns:
        probability_column = col
        break

if probability_column is None:
    raise ValueError(
        "Could not find risk probability column. "
        f"Available columns: {evaluation_intelligence.columns.tolist()}"
    )

evaluation_intelligence["risk_probability"] = pd.to_numeric(
    evaluation_intelligence[probability_column],
    errors="coerce"
)

evaluation_intelligence["priority_tier"] = (
    evaluation_intelligence["risk_probability"]
    .apply(assign_risk_tier)
)

print("Using probability column:", probability_column)

display(
    evaluation_intelligence[
        [
            "episode_id",
            "risk_probability",
            "priority_tier"
        ]
    ].head(10)
)

Using probability column: risk_probability


,episode_id,risk_probability,priority_tier
0,E0000002,0.401178,MEDIUM
1,E0000003,0.357278,MEDIUM
2,E0000009,0.563782,MEDIUM
3,E0000014,0.716091,HIGH
4,E0000018,0.687923,HIGH
5,E0000019,0.312529,MEDIUM
6,E0000021,0.355052,MEDIUM
7,E0000022,0.410201,MEDIUM
8,E0000023,0.354187,MEDIUM
9,E0000024,0.616694,HIGH


In [23]:
# CELL 13 — Explain Why the Episode Needs Attention

def generate_reason(row):

    reasons = []

    if row.get("long_distance", 0) == 1:
        reasons.append("long distance to facility")

    if row.get("medicine_required", 0) == 1:
        reasons.append("medicine collection required")

    if row.get("test_required", 0) == 1:
        reasons.append("diagnostic test required")

    if row.get("review_required", 0) == 1:
        reasons.append("follow-up review required")

    if row.get("low_digital_access", 0) == 1:
        reasons.append("low digital access")

    if row.get("elderly_living_alone", 0) == 1:
        reasons.append("elderly living alone")

    if row.get("complex_care_episode", 0) == 1:
        reasons.append("complex care requirements")

    if (
        pd.notna(row.get("previous_teleconsultations"))
        and row.get("previous_teleconsultations", 0) > 0
    ):
        reasons.append("previous teleconsultation history")

    if len(reasons) == 0:
        reasons.append("elevated predicted follow-up risk")

    return "; ".join(reasons[:4])


evaluation_intelligence["reason"] = (
    evaluation_intelligence.apply(
        generate_reason,
        axis=1
    )
)

display(
    evaluation_intelligence[
        [
            "episode_id",
            "risk_probability",
            "priority_tier",
            "reason"
        ]
    ].head(10)
)

,episode_id,risk_probability,priority_tier,reason
0,E0000002,0.401178,MEDIUM,long distance to facility; follow-up review re...
1,E0000003,0.357278,MEDIUM,long distance to facility; follow-up review re...
2,E0000009,0.563782,MEDIUM,long distance to facility; medicine collection...
3,E0000014,0.716091,HIGH,long distance to facility; medicine collection...
4,E0000018,0.687923,HIGH,long distance to facility; medicine collection...
5,E0000019,0.312529,MEDIUM,long distance to facility; diagnostic test req...
6,E0000021,0.355052,MEDIUM,long distance to facility; medicine collection...
7,E0000022,0.410201,MEDIUM,long distance to facility; diagnostic test req...
8,E0000023,0.354187,MEDIUM,long distance to facility; follow-up review re...
9,E0000024,0.616694,HIGH,long distance to facility; medicine collection...


In [24]:
# CELL 14 — Suggested Follow-up Stage

def suggest_stage(row):

    if row.get("medicine_required", 0) == 1:
        return "Medicine not collected"

    if row.get("review_required", 0) == 1:
        return "Review not attended"

    if row.get("test_required", 0) == 1:
        return "Test not completed"

    return "Care journey monitoring"


evaluation_intelligence["predicted_dropout_stage"] = (
    evaluation_intelligence.apply(
        suggest_stage,
        axis=1
    )
)

display(
    evaluation_intelligence[
        [
            "episode_id",
            "priority_tier",
            "predicted_dropout_stage"
        ]
    ].head(10)
)

,episode_id,priority_tier,predicted_dropout_stage
0,E0000002,MEDIUM,Review not attended
1,E0000003,MEDIUM,Review not attended
2,E0000009,MEDIUM,Medicine not collected
3,E0000014,HIGH,Medicine not collected
4,E0000018,HIGH,Medicine not collected
5,E0000019,MEDIUM,Review not attended
6,E0000021,MEDIUM,Medicine not collected
7,E0000022,MEDIUM,Review not attended
8,E0000023,MEDIUM,Review not attended
9,E0000024,HIGH,Medicine not collected


In [25]:

# CELL 15 — Recommended Action

def recommend_action(row):

    stage = row["predicted_dropout_stage"]

    if stage == "Medicine not collected":
        return "Verify medicine availability and coordinate collection support."

    if stage == "Review not attended":
        return "Schedule reminder and coordinate follow-up review."

    if stage == "Test not completed":
        return "Confirm test availability and coordinate completion."

    return "Conduct proactive follow-up and confirm care completion."


evaluation_intelligence["recommended_action"] = (
    evaluation_intelligence.apply(
        recommend_action,
        axis=1
    )
)

display(
    evaluation_intelligence[
        [
            "episode_id",
            "predicted_dropout_stage",
            "recommended_action"
        ]
    ].head(10)
)

,episode_id,predicted_dropout_stage,recommended_action
0,E0000002,Review not attended,Schedule reminder and coordinate follow-up rev...
1,E0000003,Review not attended,Schedule reminder and coordinate follow-up rev...
2,E0000009,Medicine not collected,Verify medicine availability and coordinate co...
3,E0000014,Medicine not collected,Verify medicine availability and coordinate co...
4,E0000018,Medicine not collected,Verify medicine availability and coordinate co...
5,E0000019,Review not attended,Schedule reminder and coordinate follow-up rev...
6,E0000021,Medicine not collected,Verify medicine availability and coordinate co...
7,E0000022,Review not attended,Schedule reminder and coordinate follow-up rev...
8,E0000023,Review not attended,Schedule reminder and coordinate follow-up rev...
9,E0000024,Medicine not collected,Verify medicine availability and coordinate co...


In [26]:
# CELL 16 — Assign Follow-up Cadre

def assign_cadre(row):

    if row["priority_tier"] == "VERY HIGH":
        return "ASHA"

    if row["predicted_dropout_stage"] in [
        "Review not attended",
        "Test not completed"
    ]:
        return "CHO"

    return "ASHA"


evaluation_intelligence["assigned_cadre"] = (
    evaluation_intelligence.apply(
        assign_cadre,
        axis=1
    )
)

display(
    evaluation_intelligence[
        [
            "episode_id",
            "priority_tier",
            "assigned_cadre"
        ]
    ].head(10)
)

,episode_id,priority_tier,assigned_cadre
0,E0000002,MEDIUM,CHO
1,E0000003,MEDIUM,CHO
2,E0000009,MEDIUM,ASHA
3,E0000014,HIGH,ASHA
4,E0000018,HIGH,ASHA
5,E0000019,MEDIUM,CHO
6,E0000021,MEDIUM,ASHA
7,E0000022,MEDIUM,CHO
8,E0000023,MEDIUM,CHO
9,E0000024,HIGH,ASHA


In [27]:
# CELL 17 — Final Action Queue

action_queue_columns = [
    "episode_id",
    "patient_id",
    "risk_probability",
    "priority_tier",
    "reason",
    "predicted_dropout_stage",
    "recommended_action",
    "assigned_cadre"
]

action_queue = evaluation_intelligence[
    action_queue_columns
].copy()

action_queue = action_queue.sort_values(
    "risk_probability",
    ascending=False
).reset_index(drop=True)

display(
    action_queue.head(20)
)

print(
    "Action queue created:",
    action_queue.shape
)

,episode_id,patient_id,risk_probability,priority_tier,reason,predicted_dropout_stage,recommended_action,assigned_cadre
0,E0000807,P000727,0.892275,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
1,E0000539,P000474,0.879615,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
2,E0005085,P004608,0.875438,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
3,E0003381,P003065,0.873450,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
4,E0003150,P002838,0.869805,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
5,E0003788,P003448,0.869635,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
6,E0001530,P001398,0.869445,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
7,E0000448,P000395,0.866064,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
8,E0002009,P001829,0.864161,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA
9,E0000171,P000149,0.863098,VERY HIGH,long distance to facility; medicine collection...,Medicine not collected,Verify medicine availability and coordinate co...,ASHA


Action queue created: (1384, 8)


In [28]:
# CELL 18 — Save Final Outputs

action_queue.to_csv(
    OUTPUT_DIR / "action_queue.csv",
    index=False
)

evaluation_intelligence.to_csv(
    OUTPUT_DIR / "evaluation_intelligence.csv",
    index=False
)

print("Saved:")
print("1. action_queue.csv")
print("2. evaluation_intelligence.csv")
print("3. feature_importance.csv")

Saved:
1. action_queue.csv
2. evaluation_intelligence.csv
3. feature_importance.csv
